In [6]:
import pandas as pd
df = pd.read_csv('../data/raw/cookie_cats.csv')
print(df.shape)
print(df.dtypes)
df.head()

(90189, 5)
userid             int64
version           object
sum_gamerounds     int64
retention_1         bool
retention_7         bool
dtype: object


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


In [7]:
# each player must appear once; duplicates would count a player twice
print('duplicate userids:', df['userid'].duplicated().sum())
# check for missing values in any column
print(df.isna().sum())

duplicate userids: 0
userid            0
version           0
sum_gamerounds    0
retention_1       0
retention_7       0
dtype: int64


In [8]:
# expect exactly two values: gate_30 (control) and gate_40 (test)
# typos or extra labels would mean broken assignment data
print(df['version'].value_counts())

version
gate_40    45489
gate_30    44700
Name: count, dtype: int64


In [9]:
# heavy right skew is normal in games; we are looking for impossible values
print(df['sum_gamerounds'].describe(percentiles=[.5, .9, .99, .999]))
# players who installed but never played a single round
print('zero rounds:', (df['sum_gamerounds'] == 0).sum())
# the most extreme players
print(df.nlargest(5, 'sum_gamerounds'))

count    90189.000000
mean        51.872457
std        195.050858
min          0.000000
50%         16.000000
90%        134.000000
99%        493.000000
99.9%     1073.624000
max      49854.000000
Name: sum_gamerounds, dtype: float64
zero rounds: 3994
        userid  version  sum_gamerounds  retention_1  retention_7
57702  6390605  gate_30           49854        False         True
7912    871500  gate_30            2961         True         True
29417  3271615  gate_40            2640         True        False
43671  4832608  gate_30            2438         True         True
48188  5346171  gate_40            2294         True         True


### Zero-round players: keep or remove?

In the previous step I found **3,994 players (about 4.4%) who played 0 rounds**.

**My decision: keep them.**

Players were put into groups when they installed the game, before they played.
So a player with 0 rounds was still part of the test. They just never played.
That is a real result, not a data error.

If I remove them, I change the groups after the test already started.
Then the difference between groups could come from my filtering,
not from moving the gate.

The 4.4% is not the reason to keep or remove them.
What matters is whether removing them changes the comparison.

Two checks before I confirm:
1. Are zero-round players split evenly between gate_30 and gate_40?
2. Does any zero-round player have retention = True? That should not be possible.

In [10]:
# zero-round players must be split roughly evenly between groups,
# otherwise the treatment may have affected who played zero
zero = df[df['sum_gamerounds'] == 0]
print(zero['version'].value_counts())

# a player with 0 rounds should not have retention = True
# if they do, the data is internally inconsistent
print(pd.crosstab(zero['retention_1'], zero['retention_7']))

version
gate_40    2057
gate_30    1937
Name: count, dtype: int64
retention_7  False  True 
retention_1              
False         3883     24
True            82      5


**Result:**
- Zero-round players are split roughly evenly: 2,057 in gate_40 (4.5%) and 1,937 in gate_30 (4.3%).
- 111 zero-round players (0.12% of all players) have retention = True.
  This is inconsistent: a player who never played should not count as retained.
  I can't tell which column is wrong, so I can't fix it.

**Decision:** keep all zero-round players, including the 111.
They are too few to change the result. I note this as a data limitation.

### Sample Ratio Mismatch (SRM) check

Players should be split 50/50 between the two groups.
The actual split is 45,489 in gate_40 and 44,700 in gate_30, a difference of 789 players.

A small difference is normal, because random assignment is never perfectly equal.
But if the difference is too big, the split itself may be broken
(for example, a bug in how players were assigned or logged).
If the split is broken, I can't trust the test results.

I use a chi-square test to check whether this difference is normal random noise.

- H0: the split is 50/50, and the difference is random noise.
- H1: the split is not 50/50.

The common rule for SRM checks: if p < 0.001, the split is broken
and needs investigation before analyzing results.

In [12]:
# expected count per group under a perfect 50/50 split
expected = len(df) / 2
print('expected per group:', expected)
print('deviation:', (df['version'].value_counts() - expected).to_dict())

expected per group: 45094.5
deviation: {'gate_40': 394.5, 'gate_30': -394.5}


In [11]:
from scipy.stats import chisquare
# SRM check: is the split consistent with a 50/50 assignment?
# industry convention: p < 0.001 means the split is broken, investigate before trusting results
observed = df['version'].value_counts().values
print(chisquare(observed))

Power_divergenceResult(statistic=np.float64(6.9024049496058275), pvalue=np.float64(0.008607987810836243))


**Result:** chi-square = 6.90, p = 0.0086

With a perfect 50/50 split, each group should have 45,094.5 players.
gate_40 has 394.5 more and gate_30 has 394.5 fewer.

The p-value is above the SRM threshold of 0.001, so the split is not considered broken,
and I continue with the analysis.

But p is below 0.05, so this difference is probably not just random noise.
I can't find the cause: the dataset has no install dates or platform info to check.
I note this as a limitation. The results should be read with some caution.

### Removing one extreme player

One player has 49,854 game rounds. The next highest is 2,961.
That is thousands of rounds per day, which is not possible for a real person.
This player also has retention_1 = False, which contradicts the huge number of rounds.
I treat this row as a data error and remove it. This is 1 player out of 90,189,
so it does not change retention, but it would push up the average number of rounds.

In [13]:
df_clean = df[df['sum_gamerounds'] < 49854].copy()
print(df.shape, '->', df_clean.shape)
df_clean.to_csv('../data/raw/cookie_cats_clean.csv', index=False)

(90189, 5) -> (90188, 5)


In [14]:
df_clean['sum_gamerounds'].describe()

count    90188.000000
mean        51.320253
std        102.682719
min          0.000000
25%          5.000000
50%         16.000000
75%         51.000000
max       2961.000000
Name: sum_gamerounds, dtype: float64